# Sample results from semi-automatic method for clinical validation
This notebooks creates a random sample for clinical validation. We include 15 cases for each manufacturer (60 in total).

In [8]:
# import libraries
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# read metadata
metadata = pd.read_csv("../../../data/raw/metadata/merged_ku_tcia.csv")

# check if subjects have scans form scanners from different manufacturer
subject_scanner_counts = metadata.groupby("new_sub_id")["Manufacturer"].nunique()
print(f"Following subjects have scans taken with scanners from different manufacturer: {len(subject_scanner_counts[subject_scanner_counts > 1])}")


# read annotations
annotator_1 = pd.read_csv("../../../data/processed/manual_annotation/annotator_1.csv")
annotator_2 = pd.read_csv("../../../data/processed/manual_annotation/annotator_2.csv")

# remove rows with empty file names
annotator_1 = annotator_1[annotator_1["File / Case ID"].notna()]
annotator_2 = annotator_2[annotator_2["File / Case ID"].notna()]

# only keep accepted or edited annotations
annotator_1 = annotator_1[annotator_1["5. Final \r\nDecision"].isin(["Accept", "Accept with edits"])]
annotator_2 = annotator_2[annotator_2["5. Final \r\nDecision"].isin(["Accept", "Accept with edits"])]

# extract subject id from file name
annotator_1["Subject ID"] = annotator_1["File / Case ID"].str.extract(r"(\d{3})")
annotator_2["Subject ID"] = annotator_2["File / Case ID"].str.extract(r"(\d{3})")

# add scanner from metadata (subxxx) to annotations (xxx)
annotator_1["Scanner"] = annotator_1["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Manufacturer"].values[0])
annotator_2["Scanner"] = annotator_2["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Manufacturer"].values[0])

# add age
annotator_1["patients_age"] = annotator_1["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Patient Age"].values[0])
annotator_2["patients_age"] = annotator_2["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Patient Age"].values[0])

# Keep only rows where "Patient Age" is between 50 and 90
annotator_1["patients_age"] = pd.to_numeric(annotator_1["patients_age"].str.extract(r"(\d+)")[0], errors="coerce")
annotator_1 = annotator_1[(annotator_1["patients_age"] >= 50) & (annotator_1["patients_age"] <= 90)]

annotator_2["patients_age"] = pd.to_numeric(annotator_2["patients_age"].str.extract(r"(\d+)")[0], errors="coerce")
annotator_2 = annotator_2[(annotator_2["patients_age"] >= 50) & (annotator_2["patients_age"] <= 90)]

# remove rows with null values in the "Scanner" column
annotator_1 = annotator_1[annotator_1["Scanner"].notnull()]
annotator_2 = annotator_2[annotator_2["Scanner"].notnull()]

# add sex
annotator_1["Patient Sex"] = annotator_1["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Patient Sex"].values[0])
annotator_2["Patient Sex"] = annotator_2["Subject ID"].apply(lambda x: metadata.loc[metadata["new_sub_id"] == f"sub{x}", "Patient Sex"].values[0])

# remove rows with sex not in ["M", "F"]
annotator_1 = annotator_1[annotator_1["Patient Sex"].isin(["M", "F"])]
annotator_2 = annotator_2[annotator_2["Patient Sex"].isin(["M", "F"])]

# concat
combined_annotations = pd.concat([annotator_2, annotator_1], ignore_index=True)

# count per scanner
scanner_counts = combined_annotations["Scanner"].value_counts()
scanner_counts

Following subjects have scans taken with scanners from different manufacturer: 0


Scanner
SIEMENS               148
GE MEDICAL SYSTEMS    113
Philips                23
TOSHIBA                15
Name: count, dtype: int64

In [50]:
# randomly sample 15 rows per scanner and save as .csv
sampled_annotations = combined_annotations.groupby('Scanner').apply(lambda x: x.sample(n=15, random_state=42)).reset_index(drop=True)
sampled_annotations = sampled_annotations.merge(
    combined_annotations[["Subject ID", "Scanner"]].drop_duplicates(),
    on="Subject ID",
    how="left"
)
sampled_annotations.to_csv("../../../data/processed/manual_annotation/samples_for_clinical_validation/sample_clinical_validation_sa_method.csv", index=False)